# Train the synthetic-document GAN in Google Colab

This notebook trains a PyTorch DCGAN using only a procedurally generated dataset of fictional demo records. No real patient data, real prescriptions, medication names, or dosing instructions are used. The generator checkpoint is saved to Google Drive; the Streamlit **Synthetic Record Lab** page loads that checkpoint and never trains on page load.

**Every output is AI-generated synthetic data, not a real prescription and not for medical use.**

In [ ]:
from pathlib import Path
import importlib
import sys
import torch
from google.colab import drive, files

drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/AI_Prescription_Analyzer")
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT = (
    Path("/content/drive/MyDrive/AI_Prescription_Analyzer")
    / "models"
    / "synthetic_record_gan.pth"
)
DRIVE_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
module_name = "synthetic_record_gan.py"
if module_name not in uploaded:
    raise ValueError(f"Upload {module_name}; received: {sorted(uploaded)}")
(MODEL_DIR / module_name).write_bytes(uploaded[module_name])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

importlib.invalidate_caches()
sys.modules.pop("models.synthetic_record_gan", None)
gan_module = importlib.import_module("models.synthetic_record_gan")
gan_module.CHECKPOINT_PATH = DRIVE_CHECKPOINT
torch.set_num_threads(4)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project files: {PROJECT_ROOT}")
print(f"Checkpoint destination: {DRIVE_CHECKPOINT}")
print(f"Training device: {DEVICE}")

In [ ]:
from models.synthetic_record_gan import SyntheticRecordDataset, generate_records, train_gan
from IPython.display import display

# A short CPU run is the default. Increase the settings for a longer run.
RECORD_COUNT = 512
EPOCHS = 5 if DEVICE == "cpu" else 20
BATCH_SIZE = 16 if DEVICE == "cpu" else 32
SEED = 42

print(f"Synthetic training records: {RECORD_COUNT:,}")
display(SyntheticRecordDataset(2, seed=SEED)[0].shape)
print(f"Checkpoint destination: {gan_module.CHECKPOINT_PATH}")

## Train generator and discriminator

The training loop uses only neutral procedural placeholder documents. The checkpoint is written to Google Drive when training completes. This notebook requires Google Colab and will not run training on website startup.

In [ ]:
history = train_gan(
    epochs=EPOCHS,
    record_count=RECORD_COUNT,
    batch_size=BATCH_SIZE,
    seed=SEED,
    progress_callback=lambda epoch, total, metrics: print(
        f"Epoch {epoch}/{total} | "
        f"Generator loss: {metrics['generator_loss']:.4f} | "
        f"Discriminator loss: {metrics['discriminator_loss']:.4f}"
    ),
)
print(f"Checkpoint saved: {gan_module.CHECKPOINT_PATH}")

In [ ]:
import pandas as pd

loss_frame = pd.DataFrame(history).set_index("epoch")
display(loss_frame)
loss_frame[["generator_loss", "discriminator_loss"]].plot(
    title="Synthetic-record DCGAN losses",
    xlabel="Epoch",
    ylabel="BCE loss",
    grid=True,
    figsize=(8, 4),
);

In [ ]:
samples = generate_records(count=4, seed=SEED)
for record, image in samples:
    print(record)
    display(image)
print(f"Checkpoint saved in Drive: {gan_module.CHECKPOINT_PATH}")

## Install the checkpoint in the website

After reviewing the raw samples, run the cell below to download the trained checkpoint. Place it in the project's `models` folder as `synthetic_record_gan.pth`. The website loads this saved file and does not train on startup.


In [ ]:
from google.colab import files

if not gan_module.CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"GAN checkpoint not found: {gan_module.CHECKPOINT_PATH}")
files.download(str(gan_module.CHECKPOINT_PATH))
